# How Far Can You Push a Frozen Backbone?

**COMS4036A / COMS7050A Computer Vision project.** Mashego Mabeloane (2654606), Basimane Ravele Lebopa (2674912)

In Lab 3 we fine-tuned an ImageNet ResNet-50 on Oxford Pets and probed it layer by layer. Pets is close to ImageNet, so that is the easy case. In this project we test how the picture changes as the target domain moves further from natural photographs, and at which depth the useful features sit in each case. We keep the backbone frozen, train linear probes at five depths, and compare against a randomly initialised network and against full end-to-end training.

We use five datasets, listed in the order we expect them to sit away from ImageNet. This order is our hypothesis, not a measured distance:

1. **Oxford-IIIT Pets**: natural photographs
2. **GTSRB**: traffic signs
3. **Satellite (4-class)**: satellite tiles
4. **C-NMC Leukemia**: microscope images of single cells
5. **Chest X-ray Pneumonia**: grayscale radiographs

We run four conditions on every dataset:

| condition | weights | what is trained | what the score reflects |
|---|---|---|---|
| `random_probe` | random | linear layer on frozen features at each depth | the architecture alone |
| `imagenet_probe` | ImageNet-1K | linear layer on frozen features at each depth | architecture + pretraining |
| `random_full` | random | the whole network | learning from scratch |
| `imagenet_ft` | ImageNet-1K | the whole network, backbone at a tenth of the head's learning rate | pretraining + adapting the features |

The notebook needs a GPU runtime. Pets and GTSRB download automatically; the other three come from Kaggle and need an API token (Section 0). A full run takes roughly 45 to 90 minutes on a T4 with the subsampling caps set below.

In [1]:
# Group members
MEMBERS = [
    # ("Student name", "Student number"),
    ("Mashego Mabeloane" , "2654606"),
    ("Basimane Ravele Lebopa", "2674912")
]
for name, number in MEMBERS:
    print(f"{number}  {name}")

2654606  Mashego Mabeloane
2674912  Basimane Ravele Lebopa


In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 0. Setup, config, and logging

In [3]:
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50

DEV = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(0)
print("device:", DEV)

device: cuda


All settings we varied are in the next cell. The caps on split sizes keep ten epochs to a few minutes per dataset; setting them to `None` uses the full datasets.

Note on the satellite data: our proposal used RSI-CB256 (35 classes), but we could not find a usable copy of it, so we use the 4-class Kaggle dataset *Satellite Image Classification* (`RSI_SLUG`) instead. The variable names `rsicb` and `RSI_SLUG` are left over from the original plan.

In [4]:
SEED = 0
DATASETS = ["pets", "gtsrb", "rsicb", "cnmc", "cxr"]  # hypothesised order of domain shift
TITLES = {"pets": "Oxford Pets", "gtsrb": "GTSRB", "rsicb": "RSI-CB256", "cnmc": "C-NMC", "cxr": "Chest X-ray"}
FLIP = {"pets": True, "gtsrb": False, "rsicb": True, "cnmc": True, "cxr": True}  # a flipped stop sign is not a stop sign

MAX_TRAIN, MAX_VAL, MAX_TEST = 4000, 1000, 2000
EPOCHS = 10
LR_HEAD, LR_BACKBONE = 1e-2, 1e-3  # as Lab 3 Section 5; random_full uses LR_HEAD for everything
CUTS = ["stem", "layer1", "layer2", "layer3", "layer4"]
PROBE_WDS = [0, 1e-4, 1e-3, 1e-2]  # probe weight decay, chosen on validation macro-F1

RSI_SLUG = "mahmoudreda55/satellite-image-classification"

USE_WANDB = True

os.makedirs("results", exist_ok=True)

In [5]:
if USE_WANDB:
    import wandb

    wandb.login()

Kaggle credentials are read from Colab Secrets (`KAGGLE_USERNAME` and `KAGGLE_KEY`, set under the key icon in the left sidebar).

In [7]:

from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("using Colab secrets")


using Colab secrets


## 1. The data

Each dataset stores its labels differently, so each one gets a small builder function that turns its folder layout into `(paths, labels)` for train, validation and test. A single `ImageDataset` class then serves all five. As in Lab 3, `__init__` stores only the file paths and `__getitem__` loads the image, so nothing large is held in memory.

Three of the datasets contain many images of the same underlying object or person. A random split would put near-copies in both train and test and inflate every score, so we split them by group:

- **GTSRB:** each physical sign is photographed about thirty times as a car approaches, so the validation split is cut by *track* (the first number in the file name).
- **C-NMC:** many cells come from each patient, so we split by patient (the second field of the file name).
- **Chest X-ray:** several radiographs per patient, so we split by patient. The official validation folder has only 16 images, so we merge it into train and re-split; the official test set is kept.
- **Pets and GTSRB** use their official test sets. **The satellite set** has no official split, so we use a stratified random 70/10/20. Neighbouring tiles may be near-duplicates, which we list as a limitation in the report.

In [ ]:
!wget -q -nc https://thor.robots.ox.ac.uk/pets/images.tar.gz
!wget -q -nc https://thor.robots.ox.ac.uk/pets/annotations.tar.gz
!tar -xzf images.tar.gz && tar -xzf annotations.tar.gz
!head -3 annotations/trainval.txt

In [ ]:
!mkdir -p data
!([ -d data/cnmc ] || kaggle datasets download -q -d andrewmvd/leukemia-classification -p data/cnmc --unzip)
!([ -d data/cxr ] || kaggle datasets download -q -d paultimothymooney/chest-xray-pneumonia -p data/cxr --unzip)
!ls data/cnmc data/cxr

We searched Kaggle for RSI-CB256. The only result returned was the 4-class *Satellite Image Classification* dataset, which is the one we use (its slug is set in `RSI_SLUG` in the config cell).

In [ ]:
!kaggle datasets list -s RSI-CB256

In [ ]:
assert "REPLACE" not in RSI_SLUG, "set RSI_SLUG in the config cell"
!([ -d data/rsicb ] || kaggle datasets download -q -d {RSI_SLUG} -p data/rsicb --unzip)
!ls data/rsicb | head

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".ppm", ".tif", ".tiff"}


def scan(root, keep=lambda p: True):
    out = []
    for dirpath, _, files in os.walk(root):
        for f in files:
            p = os.path.join(dirpath, f)
            if os.path.splitext(f)[1].lower() in IMG_EXT and "__MACOSX" not in p and keep(p):
                out.append(p)
    return sorted(out)


def split_idx(idx, y, frac, groups=None):
    # returns (rest, held_out) index arrays; stratified, or grouped when groups is given
    idx, y = np.asarray(idx), np.asarray(y)
    if groups is None:
        return train_test_split(idx, test_size=frac, stratify=y[idx], random_state=SEED)
    a, b = next(GroupShuffleSplit(1, test_size=frac, random_state=SEED).split(idx, y[idx], np.asarray(groups)[idx]))
    return idx[a], idx[b]


def pack(paths, y, classes, tr, va, te):
    paths, y = np.asarray(paths), np.asarray(y)
    return {"train": (paths[tr], y[tr]), "val": (paths[va], y[va]), "test": (paths[te], y[te]), "classes": classes}


def subsample(paths, y, n):
    if n is None or len(paths) <= n:
        return paths, y
    keep, _ = train_test_split(np.arange(len(paths)), train_size=n, stratify=y, random_state=SEED)
    return paths[keep], y[keep]


def folder_labels(paths):
    names = sorted({os.path.basename(os.path.dirname(p)) for p in paths})
    return np.array([names.index(os.path.basename(os.path.dirname(p))) for p in paths]), names

Pets is parsed the same way as in Lab 3: read the official split files, take the image stem and class ID, and strip the trailing number from the stem to get the breed name. The official trainval set is split 85/15 into train and validation.

In [ ]:
def build_pets():
    def read(name):
        stems, labels = [], []
        for line in open(f"annotations/{name}.txt"):
            stem, cid = line.split()[:2]
            stems.append(stem)
            labels.append(int(cid) - 1)
        return stems, labels

    tv_stems, tv_y = read("trainval")
    te_stems, te_y = read("test")
    classes = [None] * 37
    for s, y in zip(tv_stems + te_stems, tv_y + te_y):
        classes[y] = s.rsplit("_", 1)[0]
    paths = [f"images/{s}.jpg" for s in tv_stems + te_stems]
    y = tv_y + te_y
    n_tv = len(tv_stems)
    tr, va = split_idx(np.arange(n_tv), y, 0.15)
    return pack(paths, y, classes, tr, va, np.arange(n_tv, len(paths)))


def build_gtsrb():
    from torchvision.datasets import GTSRB

    tv = GTSRB("data", split="train", download=True)
    te = GTSRB("data", split="test", download=True)
    samples = tv._samples + te._samples
    paths = [s[0] for s in samples]
    y = [s[1] for s in samples]
    n_tv = len(tv._samples)
    tracks = [f"{y[i]}_{os.path.basename(paths[i]).split('_')[0]}" for i in range(n_tv)]
    tr, va = split_idx(np.arange(n_tv), y, 0.15, tracks + ["x"] * len(te._samples))
    return pack(paths, y, [str(i) for i in range(43)], tr, va, np.arange(n_tv, len(paths)))


def build_rsicb():
    paths = scan("data/rsicb")
    y, classes = folder_labels(paths)
    print("RSI-CB classes found:", len(classes), "(expected 35)")
    rest, te = split_idx(np.arange(len(paths)), y, 0.2)
    tr, va = split_idx(rest, y, 0.125)
    return pack(paths, y, classes, tr, va, te)


def build_cnmc():
    paths = scan("data/cnmc", keep=lambda p: "training_data" in p)  # only the labelled folds
    y, classes = folder_labels(paths)
    patients = [os.path.basename(p).split("_")[1] for p in paths]
    rest, te = split_idx(np.arange(len(paths)), y, 0.2, patients)
    tr, va = split_idx(rest, y, 0.125, patients)
    return pack(paths, y, classes, tr, va, te)


def build_cxr():
    seen, paths, where = set(), [], []
    for p in scan("data/cxr"):
        parts = p.split(os.sep)
        sp = next((s for s in ("train", "val", "test") if s in parts), None)
        key = (sp, os.path.basename(os.path.dirname(p)), os.path.basename(p))
        if sp is None or key in seen:  # the archive holds a nested duplicate copy
            continue
        seen.add(key)
        paths.append(p)
        where.append(sp)
    y, classes = folder_labels(paths)
    where = np.array(where)
    patients = [os.path.basename(p).split("_")[0] if os.path.basename(p).startswith("person")
                else os.path.basename(p).rsplit("-", 1)[0] for p in paths]
    tr, va = split_idx(np.where(where != "test")[0], y, 0.1, patients)
    return pack(paths, y, classes, tr, va, np.where(where == "test")[0])


BUILDERS = dict(pets=build_pets, gtsrb=build_gtsrb, rsicb=build_rsicb, cnmc=build_cnmc, cxr=build_cxr)

The dataset class. One addition compared with Lab 3: `img.draft` tells the JPEG decoder to decode at reduced size when the file is much larger than needed. The chest X-rays are over a thousand pixels on a side, and without this the two data-loading workers could not keep the GPU busy. It has no effect on non-JPEG files.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = list(paths)
        self.labels = [int(l) for l in labels]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i])
        img.draft("RGB", (320, 320))
        img = img.convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[i]

Here we build the splits for all five datasets (GTSRB is about 300 MB to download), apply the size caps with stratified subsampling, and print a summary: split sizes, number of classes, and the smallest and largest class in the training split. This is Table I in the report.

In [ ]:
SPLITS = {}
for name in DATASETS:
    s = BUILDERS[name]()
    for split, cap in [("train", MAX_TRAIN), ("val", MAX_VAL), ("test", MAX_TEST)]:
        s[split] = subsample(*s[split], cap)
    SPLITS[name] = s

rows = []
for name in DATASETS:
    s = SPLITS[name]
    counts = np.bincount(s["train"][1], minlength=len(s["classes"]))
    rows.append(dict(dataset=TITLES[name], train=len(s["train"][0]), val=len(s["val"][0]), test=len(s["test"][0]),
                     classes=len(s["classes"]), min_per_class=counts.min(), max_per_class=counts.max()))
pd.DataFrame(rows).set_index("dataset")

## 2. Transforms

We use the same two pipelines as Lab 3. The evaluation pipeline resizes the short side to 256, centre-crops to 224 and applies ImageNet normalisation. The training pipeline adds a random resized crop (scale 0.6 to 1.0) and a horizontal flip, except for GTSRB, where a mirrored sign can mean a different sign. ImageNet statistics are used for every dataset, including the X-rays, because those are the statistics the pretrained weights expect; the random network gets the same inputs so the two are comparable.

The centre crop is square, so non-square images (the X-rays in particular) lose their edges. We report this as a design choice.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def make_loaders(splits, flip):
    norm = transforms.Normalize(MEAN, STD)
    eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), norm])
    aug = [transforms.RandomResizedCrop(224, scale=(0.6, 1.0))]
    if flip:
        aug.append(transforms.RandomHorizontalFlip())
    train_tf = transforms.Compose(aug + [transforms.ToTensor(), norm])

    sets = {
        "train": ImageDataset(*splits["train"], train_tf),
        "train_eval": ImageDataset(*splits["train"], eval_tf),  # for features: same image, same feature, every time
        "val": ImageDataset(*splits["val"], eval_tf),
        "test": ImageDataset(*splits["test"], eval_tf),
    }
    return {k: DataLoader(v, batch_size=32 if k == "train" else 64, shuffle=(k == "train"),
                          drop_last=(k == "train"), num_workers=2) for k, v in sets.items()}

One augmented training batch per dataset, with the normalisation undone for display. We used this to check that the crops keep the object and the colours are correct (Fig. 1 in the report).

In [ ]:
def show_batch(loader, ax_row, n=8):
    x, y = next(iter(loader))
    x = (x[:n] * torch.tensor(STD).view(1, 3, 1, 1) + torch.tensor(MEAN).view(1, 3, 1, 1)).clamp(0, 1)
    for ax, img in zip(ax_row, x):
        ax.imshow(img.permute(1, 2, 0))
        ax.axis("off")


fig, axes = plt.subplots(len(DATASETS), 8, figsize=(16, 2.1 * len(DATASETS)))
for ax_row, name in zip(axes, DATASETS):
    show_batch(make_loaders(SPLITS[name], FLIP[name])["train"], ax_row)
    ax_row[0].set_title(TITLES[name], loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## 3. The models

`make_model(pretrained, n_classes)` returns a ResNet-50 with a new `fc` layer sized for the dataset. With `pretrained=True` the backbone has the torchvision ImageNet-1K weights; with `False` it is the same architecture at random initialisation. Each model is used in two ways: as a frozen feature extractor for the probes (Section 4, where `fc` is not used) and as a network trained end to end (Section 5).

In [ ]:
def make_model(pretrained, n_classes):
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
    model.fc = nn.Linear(2048, n_classes)
    return model


m = make_model(True, 10)
print(f"{sum(p.numel() for p in m.parameters()):,} parameters")

## 4. Linear probes at five depths

`extract_features` runs the frozen backbone once over a split and returns the globally average-pooled activations at five cuts: the **stem** (conv1, batch norm, ReLU and max-pool; 64 channels) and the outputs of `layer1` to `layer4` (256, 512, 1024 and 2048 channels). All five cuts come from the same forward pass. The backbone stays in `eval()` mode, so batch norm uses its stored statistics; for the random network these are the initial values (mean 0, variance 1).

The feature dimension grows with depth, from 64 to 2048, so a probe at `layer4` has more parameters than one at the stem. Part of any trend with depth is therefore due to dimension rather than representation quality.

In [ ]:
@torch.no_grad()
def extract_features(model, loader):
    model.eval().to(DEV)
    feats = {c: [] for c in CUTS}
    labels = []
    for x, y in loader:
        x = x.to(DEV)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
            h = model.maxpool(model.relu(model.bn1(model.conv1(x))))
            out = {"stem": h}
            h = model.layer1(h)
            out["layer1"] = h
            h = model.layer2(h)
            out["layer2"] = h
            h = model.layer3(h)
            out["layer3"] = h
            h = model.layer4(h)
            out["layer4"] = h
        for c in CUTS:
            feats[c].append(out[c].float().mean((2, 3)).cpu())
        labels.append(y)
    return {c: torch.cat(v) for c, v in feats.items()}, torch.cat(labels)

`fit_probe` standardises the features using the training set's mean and standard deviation, then trains a single `nn.Linear` layer with Adam (learning rate 1e-3, batch 256, 100 epochs) on the cached features. Weight decay is chosen from `PROBE_WDS` using **validation** macro-F1, and the test set is evaluated only for the chosen value. We report macro-F1 as the main metric because GTSRB, C-NMC and Chest X-ray are class-imbalanced, and accuracy alone rewards predicting the majority class; test accuracy is stored as well.

In [ ]:
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")


def fit_probe(train, val, test, n_classes, epochs=100):
    (ftr, ytr), (fva, yva), (fte, yte) = [(f.to(DEV), y.to(DEV)) for f, y in (train, val, test)]
    mu, sd = ftr.mean(0, keepdim=True), ftr.std(0, keepdim=True) + 1e-6
    ftr, fva, fte = (ftr - mu) / sd, (fva - mu) / sd, (fte - mu) / sd

    best = None
    for wd in PROBE_WDS:
        seed_everything(SEED)
        lin = nn.Linear(ftr.shape[1], n_classes).to(DEV)
        opt = torch.optim.Adam(lin.parameters(), lr=1e-3, weight_decay=wd)
        for _ in range(epochs):
            perm = torch.randperm(len(ftr), device=DEV)
            for i in range(0, len(ftr), 256):
                idx = perm[i:i + 256]
                loss = F.cross_entropy(lin(ftr[idx]), ytr[idx])
                opt.zero_grad()
                loss.backward()
                opt.step()
        with torch.no_grad():
            va_pred = lin(fva).argmax(1).cpu().numpy()
            va_f1 = macro_f1(yva.cpu().numpy(), va_pred)
            if best is None or va_f1 > best["val_f1"]:
                te_pred = lin(fte).argmax(1).cpu().numpy()
                best = dict(val_f1=va_f1, test_acc=float((te_pred == yte.cpu().numpy()).mean()),
                            test_f1=macro_f1(yte.cpu().numpy(), te_pred), wd=wd)
    return best

## 5. End-to-end training

`evaluate` and `train` are adapted from our Lab 3 code, with three changes: `evaluate` also returns macro-F1; `train` takes the dictionary of loaders from Section 2; and `train` keeps the weights from the epoch with the best **validation** macro-F1 and reports the test score for those weights, so no epoch is selected using the test set.

`make_optimizer` uses the Lab 3 recipe: SGD with momentum 0.9, weight decay $10^{-4}$, and two parameter groups. For `imagenet_ft` the backbone learning rate is a tenth of the head's ($10^{-3}$ vs $10^{-2}$), so large updates do not destroy the pretrained weights. For `random_full` there are no pretrained weights to protect, so the whole network uses $10^{-2}$. These settings were tuned on Pets in Lab 3 and are not re-tuned per dataset.

In [ ]:
def evaluate(model, loader):
    model.eval()
    total, preds, ys = 0.0, [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEV), y.to(DEV)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
                out = model(x)
            total += F.cross_entropy(out.float(), y, reduction="sum").item()
            preds.append(out.argmax(1).cpu())
            ys.append(y.cpu())
    preds, ys = torch.cat(preds).numpy(), torch.cat(ys).numpy()
    return total / len(ys), float((preds == ys).mean()), macro_f1(ys, preds)


def make_optimizer(model, pretrained):
    backbone = [p for n, p in model.named_parameters() if not n.startswith("fc.")]
    lr_backbone = LR_BACKBONE if pretrained else LR_HEAD
    return torch.optim.SGD([{"params": backbone, "lr": lr_backbone},
                            {"params": model.fc.parameters(), "lr": LR_HEAD}],
                           momentum=0.9, weight_decay=1e-4)


def train(model, epochs, opt, run_name, config, loaders):
    if USE_WANDB:
        wandb.init(project="cv-project", name=run_name, config=config, reinit=True)
    scaler = torch.amp.GradScaler(enabled=DEV == "cuda")
    history, best_f1, best_state = [], -1, None
    for ep in range(epochs):
        model.train()
        tot_loss, correct, n = 0.0, 0, 0
        for x, y in loaders["train"]:
            x, y = x.to(DEV), y.to(DEV)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
                out = model(x)
                loss = F.cross_entropy(out.float(), y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
        va_loss, va_acc, va_f1 = evaluate(model, loaders["val"])
        rec = {"epoch": ep + 1, "train/loss": tot_loss / n, "train/acc": correct / n,
               "val/loss": va_loss, "val/acc": va_acc, "val/f1": va_f1}
        history.append(rec)
        if USE_WANDB:
            wandb.log(rec)
        print(f"  ep {ep + 1:2d}  train {rec['train/loss']:.3f} / {rec['train/acc']:.3f}   "
              f"val {va_loss:.3f} / {va_acc:.3f}   val F1 {va_f1:.3f}")
        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    _, va_acc, va_f1 = evaluate(model, loaders["val"])
    _, te_acc, te_f1 = evaluate(model, loaders["test"])
    if USE_WANDB:
        wandb.summary.update({"test/acc": te_acc, "test/f1": te_f1})
        wandb.finish()
    return history, dict(val_f1=va_f1, test_acc=te_acc, test_f1=te_f1)

## 6. Running the four conditions on every dataset

For each dataset we build the loaders, probe the random and the ImageNet backbone at all five cuts, and then train `random_full` and `imagenet_ft` for `EPOCHS` epochs. Every run is seeded with `SEED`. Results are appended to `results/results.csv` after each dataset, so if the Colab session disconnects, re-running this cell skips the datasets already finished and continues from the next one.

In [ ]:
RES_FILE = "results/results.csv"
results = pd.read_csv(RES_FILE).to_dict("records") if os.path.exists(RES_FILE) else []
done = {r["dataset"] for r in results}


def run_dataset(name):
    splits = SPLITS[name]
    n_classes = len(splits["classes"])
    loaders = make_loaders(splits, FLIP[name])
    new = []

    for cond, pretrained in [("random_probe", False), ("imagenet_probe", True)]:
        seed_everything(SEED)
        model = make_model(pretrained, n_classes)
        feats, labels = {}, {}
        for split, key in [("train", "train_eval"), ("val", "val"), ("test", "test")]:
            feats[split], labels[split] = extract_features(model, loaders[key])
        for cut in CUTS:
            r = fit_probe(*[(feats[s][cut], labels[s]) for s in ("train", "val", "test")], n_classes)
            new.append(dict(dataset=name, condition=cond, layer=cut, **r))
            print(f"  {cond:15s} {cut:7s} dim {feats['train'][cut].shape[1]:4d}   "
                  f"test acc {r['test_acc']:.3f}   F1 {r['test_f1']:.3f}")
        del model, feats

    for cond, pretrained in [("random_full", False), ("imagenet_ft", True)]:
        print(f"  -- {cond}")
        seed_everything(SEED)
        model = make_model(pretrained, n_classes).to(DEV)
        opt = make_optimizer(model, pretrained)
        config = dict(dataset=name, recipe=cond, pretrained=pretrained, epochs=EPOCHS, seed=SEED,
                      lr_head=LR_HEAD, lr_backbone=LR_BACKBONE if pretrained else LR_HEAD,
                      train_size=len(splits["train"][0]),
                      trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad))
        _, r = train(model, EPOCHS, opt, f"{name}-{cond}", config, loaders)
        new.append(dict(dataset=name, condition=cond, layer="all", **r))
        print(f"  {cond}: test acc {r['test_acc']:.3f}   F1 {r['test_f1']:.3f}")
        del model, opt
        torch.cuda.empty_cache()
    return new


for name in DATASETS:
    if name in done:
        print(f"== {TITLES[name]}: already done")
        continue
    t0 = time.time()
    print(f"\n===== {TITLES[name]} =====")
    results += run_dataset(name)
    pd.DataFrame(results).to_csv(RES_FILE, index=False)
    print(f"{TITLES[name]} finished in {(time.time() - t0) / 60:.1f} min")

## 7. Results

### Depth profiles (Fig. 2, Table II)

Test macro-F1 at each cut for the random and ImageNet probes, one panel per dataset. The two end-to-end results have no depth axis, so they are drawn as horizontal dashed lines for reference. Test accuracy is also stored in `results.csv`.

In [ ]:
df = pd.read_csv(RES_FILE)


def plot_depth_curves(df, metric="test_f1", ylabel="Test macro-F1"):
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(4.2 * len(DATASETS), 3.8), sharey=True)
    for ax, name in zip(axes, DATASETS):
        sub = df[df.dataset == name]
        for cond, label, color in [("random_probe", "random + probe", "tab:gray"),
                                   ("imagenet_probe", "ImageNet + probe", "tab:blue")]:
            s = sub[sub.condition == cond].set_index("layer").loc[CUTS]
            ax.plot(CUTS, s[metric], marker="o", color=color, label=label)
        for cond, label, color in [("random_full", "random, full training", "tab:orange"),
                                   ("imagenet_ft", "ImageNet, fine-tuned", "tab:green")]:
            ax.axhline(sub[sub.condition == cond][metric].iloc[0], ls="--", color=color, label=label)
        ax.set_title(TITLES[name])
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.3)
        ax.tick_params(axis="x", rotation=30)
    axes[0].set_ylabel(ylabel)
    axes[-1].legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.show()


plot_depth_curves(df)

### Best-cut summary (Table III)

For each dataset we select the best cut of each probe using **validation** F1, so the choice does not look at the test set, and report the test macro-F1 at that cut. The probe gap is the ImageNet probe's score minus the random probe's score at their respective best cuts. The table also lists the two end-to-end results and the gain from fine-tuning over training from scratch. The summary is saved to `results/summary.csv`.

In [ ]:
rows = []
for name in DATASETS:
    sub = df[df.dataset == name]
    rec = {"dataset": TITLES[name]}
    for cond, tag in [("imagenet_probe", "imagenet"), ("random_probe", "random")]:
        s = sub[sub.condition == cond].set_index("layer").loc[CUTS]
        peak = s["val_f1"].idxmax()
        rec[f"{tag}_peak"] = peak
        rec[f"{tag}_F1"] = s.loc[peak, "test_f1"]
    rec["probe_gap"] = rec["imagenet_F1"] - rec["random_F1"]
    rec["scratch_F1"] = sub[sub.condition == "random_full"].test_f1.iloc[0]
    rec["finetune_F1"] = sub[sub.condition == "imagenet_ft"].test_f1.iloc[0]
    rec["finetune_gain"] = rec["finetune_F1"] - rec["scratch_F1"]
    rows.append(rec)
summary = pd.DataFrame(rows).set_index("dataset")
summary.to_csv("results/summary.csv")
summary.round(3)

### Value of pretraining (Fig. 3)

Left: the ImageNet probe minus the random probe at each cut, one line per dataset, which shows at which depth pretraining helps. Right: two summary measures of the value of ImageNet per dataset, the best-cut probe gap and fine-tune minus scratch.

In [ ]:
f1 = df.pivot_table(index=["dataset", "condition"], columns="layer", values="test_f1")
gap = (f1.xs("imagenet_probe", level="condition") - f1.xs("random_probe", level="condition"))[CUTS].loc[DATASETS]
gap.index = [TITLES[d] for d in gap.index]
print(gap.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for name in DATASETS:
    axes[0].plot(CUTS, gap.loc[TITLES[name]], marker="o", label=TITLES[name])
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_title("ImageNet minus random probe, by depth")
axes[0].set_ylabel("test macro-F1 gap")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

x = np.arange(len(DATASETS))
axes[1].bar(x - 0.2, summary["probe_gap"], 0.4, label="frozen probe, best cut")
axes[1].bar(x + 0.2, summary["finetune_gain"], 0.4, label="fine-tune minus scratch")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary.index, rotation=20)
axes[1].set_title("Value of ImageNet, by domain")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()